In [4]:
# THE PLAN:
#    Send a ADQL query to the Gaia archive, 
#    asking it to return all the stars in a small patch of sky around NGC 2244,
#    Then, trimming down the data set for qualifying canadates

#    NGC 2244    RA: 06h 32m 18s , DEC +04° 52′ 00″

#    NGC 2244 itself spans about 30 arcminutes (0.5°), 
#        So we may want to search radius around 1 degree, 
#        giving us a generous field around the cluster 
#        this would give field-star background for contrast


In [6]:
pip install astroquery

  Obtaining dependency information for astroquery from https://files.pythonhosted.org/packages/e3/ca/944b328f2c60b83896a9223312e21e46cdc1fef57565e853da4544ff6a8e/astroquery-0.4.11-py3-none-any.whl.metadata
  Obtaining dependency information for html5lib>=0.999 from https://files.pythonhosted.org/packages/6c/dd/a834df6482147d48e225a49515aabc28974ad5a4ca3215c18a882565b028/html5lib-1.1-py2.py3-none-any.whl.metadata
  Using cached html5lib-1.1-py2.py3-none-any.whl.metadata (16 kB)
  Obtaining dependency information for keyring>=15.0 from https://files.pythonhosted.org/packages/81/db/e655086b7f3a705df045bf0933bdd9c2f79bb3c97bfef1384598bb79a217/keyring-25.7.0-py3-none-any.whl.metadata
  Obtaining dependency information for pyvo>=1.5 from https://files.pythonhosted.org/packages/b8/e8/37bad20ef0215b13ace46f1f9a32848511fb7a4bd0c06fff9e63070e14a8/pyvo-1.8.1-py3-none-any.whl.metadata
  Obtaining dependency information for jaraco.classes from https://files.pythonhosted.org/packages/7f/66/b15ce6255

In [18]:
from astropy.table import Table
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import astroquery
from astroquery.gaia import Gaia

In [8]:
tables = Gaia.load_tables()

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


In [10]:
#    These are the table names from our dataset
for i, tab in enumerate(tables):
    print (i, tab.name)

0 external.apassdr9
1 external.catwise2020
2 external.gaiadr2_astrophysical_parameters
3 external.gaiadr2_geometric_distance
4 external.gaiaedr3_distance
5 external.gaiaedr3_gcns_main_1
6 external.gaiaedr3_gcns_rejected_1
7 external.gaiaedr3_spurious
8 external.gaia_eso_survey
9 external.galex_ais
10 external.lamost_dr9_lrs
11 external.lamost_dr9_mrs
12 external.ravedr5_com
13 external.ravedr5_dr5
14 external.ravedr5_gra
15 external.ravedr5_on
16 external.ravedr6
17 external.sdssdr13_photoprimary
18 external.skymapperdr1_master
19 external.skymapperdr2_master
20 external.tmass_xsc
21 external.xgboost_table1
22 external.xgboost_table2
23 gaiadr1.aux_qso_icrf2_match
24 gaiadr1.ext_phot_zero_point
25 gaiadr1.allwise_best_neighbour
26 gaiadr1.allwise_neighbourhood
27 gaiadr1.gsc23_best_neighbour
28 gaiadr1.gsc23_neighbourhood
29 gaiadr1.ppmxl_best_neighbour
30 gaiadr1.ppmxl_neighbourhood
31 gaiadr1.sdss_dr9_best_neighbour
32 gaiadr1.sdss_dr9_neighbourhood
33 gaiadr1.tmass_best_neighbour
34

In [12]:
#    We're interested in the "gaiadr3.gaia_source" table, this is entry 94. 
#    We can also see what columns are included in this table:

In [13]:
cols = tables[94].columns
for col in cols:
    print (col.name)

solution_id
designation
source_id
random_index
ref_epoch
ra
ra_error
dec
dec_error
parallax
parallax_error
parallax_over_error
pm
pmra
pmra_error
pmdec
pmdec_error
ra_dec_corr
ra_parallax_corr
ra_pmra_corr
ra_pmdec_corr
dec_parallax_corr
dec_pmra_corr
dec_pmdec_corr
parallax_pmra_corr
parallax_pmdec_corr
pmra_pmdec_corr
astrometric_n_obs_al
astrometric_n_obs_ac
astrometric_n_good_obs_al
astrometric_n_bad_obs_al
astrometric_gof_al
astrometric_chi2_al
astrometric_excess_noise
astrometric_excess_noise_sig
astrometric_params_solved
astrometric_primary_flag
nu_eff_used_in_astrometry
pseudocolour
pseudocolour_error
ra_pseudocolour_corr
dec_pseudocolour_corr
parallax_pseudocolour_corr
pmra_pseudocolour_corr
pmdec_pseudocolour_corr
astrometric_matched_transits
visibility_periods_used
astrometric_sigma5d_max
matched_transits
new_matched_transits
matched_transits_removed
ipd_gof_harmonic_amplitude
ipd_gof_harmonic_phase
ipd_frac_multi_peak
ipd_frac_odd_win
ruwe
scan_direction_strength_k1
scan_di

In [14]:
#    Such Cool Much WoW

In [15]:
#    Here is my example query for
#        Finding the 100 brightest stars anywhere in the entire sky
#        that are within 200 parsecs of Earth with some quality cuts


#query_size = 100  # Number of stars we want to get was 100000
#distance = 200  # Distance (in pc) out to which we will query
#job = Gaia.launch_job_async("select top {}".format(query_size)+
#                 " ra, dec, parallax, parallax_over_error, "   # Getting source location and parallax
#                 " bp_rp, phot_g_mean_mag "                    # Getting color and magnitude measurements
#                 " from gaiadr3.gaia_source"                   # Selecting the data source
#                 # All of these are data quality checks
#                 " where parallax_over_error > 10"
#                 " and visibility_periods_used > 8"
#                 " and phot_g_mean_flux_over_error > 50"
#                 " and phot_bp_mean_flux_over_error > 20"
#                 " and phot_rp_mean_flux_over_error > 20"
#                 " and phot_bp_rp_excess_factor <"
#                 " 1.3+0.06*power(phot_bp_mean_mag-phot_rp_mean_mag,2)"
#                 " and phot_bp_rp_excess_factor >"
#                 " 1.0+0.015*power(phot_bp_mean_mag-phot_rp_mean_mag,2)"
#                 " and astrometric_chi2_al/(astrometric_n_good_obs_al-5)<"
#                 "1.44*greatest(1,exp(-0.4*(phot_g_mean_mag-19.5)))"
#                 # Filtering on distance
#                 +" and 1000/parallax <= {}".format(distance))


#r = job.get_results()
##  Convert to pandas
#df = r.to_pandas()
# # Save to a csv
#df.to_csv("gaia3.csv")

INFO: Query finished. [astroquery.utils.tap.core]


In [ ]:
#    Lets rewrite this query to do a cone search around NGC 2244,
#    include the source id, mra, pmdec, ruwe), andmra, pmdec, ruwe),
#    and uses phot_g_mean_mag < 18, parallax_over_error > 5, ruwe < 1.4
#        Cone Searches:
#            ADQL has a function for cone searches,
#            CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS',
#            center_ra, center_dec, radius)) = 1
#
#        Here we give the star's position as POINT,
#        take a circle (using center coordinates and the radius in degrees,
#        It will return 1 if the star is inside the circle


In [ ]:
job = Gaia.launch_job_async("select"
                 " ra, dec, parallax, parallax_over_error, "   # Getting source location and parallax
                 " bp_rp, phot_g_mean_mag, "                   # Getting color and magnitude measurements
                 " source_id, pmra, pmdec, ruwe "              # Getting source_id, proper motion and ruwe
                 " from gaiadr3.gaia_source"                   # Selecting the data source
                 # All of these are data quality checks
                 " where parallax_over_error > 5"
                 " and CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 98.075, 4.867, 1.0)) = 1"
                 " and visibility_periods_used > 8"
                 " and phot_g_mean_mag < 18"
                 " and ruwe < 1.4"
                 #" and phot_g_mean_flux_over_error > 18"    # maybe remove, as ruwe < 1.4 cleans data
                 #" and phot_bp_mean_flux_over_error > 20"   # maybe remove, as ruwe < 1.4 cleans data
                 #" and phot_rp_mean_flux_over_error > 20"  # maybe remove, as ruwe < 1.4 cleans data
                           )


r = job.get_results()
    #  Convert to pandas
df = r.to_pandas()
    # Save to a csv
df.to_csv("gaia3.csv")

In [17]:
import pandas as pd
df = pd.read_csv("gaia3.csv")
df.head()

,Unnamed: 0,ra,dec,parallax,parallax_over_error,bp_rp,phot_g_mean_mag
0,0,45.010277,0.351101,5.953794,409.47415,0.682347,10.017940
1,1,44.673246,0.442521,5.005910,63.06303,2.428698,16.852789
2,2,45.782575,1.299327,5.682378,47.54824,2.686440,17.405735
3,3,44.372422,1.084423,5.469907,438.72330,1.127535,12.324314
4,4,44.520431,1.593550,5.522669,388.89185,1.355875,13.144315
